# LearnSphere AI - HLR Baseline Notebook

Bu notebook Sprint 1 veri bilimi görevini gösterir:

1. Gerçek kullanıcı verisi yokken sentetik öğrenme/quiz logları üretmek
2. Half-Life Regression (HLR) benzeri unutma modeli için feature engineering planını kurmak
3. Baseline modeli eğitmek ve metrikleri görmek
4. Quiz ajanına gönderilebilecek `unutma riski yüksek konu` listesini üretmek

## Model Fikri

LearnSphere kullanıcının çalıştığı konuları takip eder. Veri bilimi tarafında amaç, kullanıcının bir konuyu ne zaman unutmaya yaklaşacağını tahmin etmektir.

Basit formül:

```text
recall_probability = 2 ^ (-elapsed_days / half_life)
half_life = exp(feature_weights * features)
```

`recall_probability` düştüğünde sistem o konu için quiz önerebilir.

## Kullanılan Feature'lar

- `difficulty`: konunun zorluğu
- `elapsed_days`: son görülmeden bu yana geçen gün
- `prior_exposures`: konunun daha önce kaç kez görüldüğü
- `successful_recalls`: önceki başarılı quiz sayısı
- `failed_recalls`: önceki başarısız quiz sayısı
- `prior_accuracy`: geçmiş başarı oranı
- `motivation`: sentetik kullanıcı motivasyonu
- `prior_knowledge`: sentetik ön bilgi seviyesi

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

cwd = Path.cwd()
project_dir = cwd.parent if cwd.name == "notebooks" else (cwd / "data-science" if (cwd / "data-science").exists() else cwd)
project_dir

## 1. Sentetik Veri Üretimi

100 hayali öğrenci için 30 günlük öğrenme ve quiz geçmişi oluşturulur.

In [ ]:
subprocess.run([
    sys.executable,
    str(project_dir / "scripts" / "generate_synthetic_data.py"),
    "--output-dir",
    str(project_dir / "data"),
], check=True)

In [ ]:
summary_path = project_dir / "data" / "dataset_summary.txt"
print(summary_path.read_text(encoding="utf-8"))

## 2. Baseline Model Eğitimi

İki yaklaşım karşılaştırılır:

- Rule-based baseline: sabit Ebbinghaus/spaced repetition kuralları
- HLR-inspired model: half-life değerini feature'lardan öğrenen küçük model

In [ ]:
subprocess.run([
    sys.executable,
    str(project_dir / "scripts" / "train_forgetting_baseline.py"),
    "--data-dir",
    str(project_dir / "data"),
    "--output-dir",
    str(project_dir / "outputs"),
], check=True)

In [ ]:
metrics = json.loads((project_dir / "outputs" / "model_metrics.json").read_text(encoding="utf-8"))
metrics

## 3. Unutma Riski Yüksek Konular

Bu çıktı Sprint 3'te quiz ajanına bağlanabilir. Örneğin: `user_091` kullanıcısı `rag` konusunu unutmak üzereyse RAG destekli quiz ajanı bu konu için soru üretebilir.

In [ ]:
import pandas as pd

recommendations = pd.read_csv(project_dir / "outputs" / "at_risk_recommendations.csv")
recommendations.head(10)

## 4. Modelin Görselleştirilmesi: Unutma Eğrisi (Forgetting Curve)

Modelimizin temel aldığı HLR (Half-Life Regression) mantığını görselleştirelim.
Farklı yarı ömürlere (half-life) sahip konuların zaman içinde nasıl unutulduğunu aşağıda görebiliriz.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 30 günlük bir zaman çizelgesi oluşturalım
days = np.linspace(0, 30, 100)

# Farklı yarı ömür değerleri (Örn: Zor konu 2 gün, kolay konu 14 gün)
half_lives = [2, 5, 10, 20]
colors = ['#e63946', '#f4a261', '#2a9d8f', '#264653']
labels = ['Zor/Yeni Konu (h=2)', 'Orta Konu (h=5)', 'Pekiştirilmiş Konu (h=10)', 'Çok İyi Bilinen Konu (h=20)']

plt.figure(figsize=(10, 6))

for h, color, label in zip(half_lives, colors, labels):
    # Formül: P = 2^(-t/h)
    prob = 2 ** (-days / h)
    plt.plot(days, prob, label=label, color=color, linewidth=2.5)

# Yüzde 50 unutma sınırını çizelim (Quiz Ajanı'nın harekete geçeceği eşik)
plt.axhline(y=0.5, color='gray', linestyle='--', alpha=0.7, label='%50 Hatırlama Sınırı (Quiz Zamanı)')

plt.title('LearnSphere - HLR Unutma Eğrisi (Forgetting Curve)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Son Etkileşimden Sonra Geçen Gün Sayısı', fontsize=12)
plt.ylabel('Hatırlama İhtimali', fontsize=12)
plt.yticks([0, 0.2, 0.4, 0.5, 0.6, 0.8, 1.0], ['0%', '20%', '40%', '50%', '60%', '80%', '100%'])
plt.legend(title='Konu Durumu (Yarı Ömür)', loc='upper right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Sprint 1 Sonucu

Bu notebook ile veri bilimi ekibi şunları kanıtlamış olur:

- Mock veri üretilebiliyor.
- Modelin kullanacağı feature seti belli.
- İlk train denemesi yapılabiliyor.
- Baseline metrikler ölçülebiliyor.
- Quiz ajanına gidecek riskli konu listesi üretilebiliyor.